# Five quantum dots, one impurity, and a field

Analysis of the COMSOL eigenvalue exports for a five-dot GaAs/AlGaAs chain.
The single table `data/eigenvalues.csv` already contains the sweep value, the
mode index and the eigenvalue, so every result below is one filter away.

This notebook reproduces the three effects of the thesis:

1. impurity position (uniform and non-uniform chain);
2. degree of non-uniformity (`R5`), with and without impurity;
3. electric field along the chain axis.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display

from qdchain import case, fit_two_lines, load, modes, plots

df = load()
print(f"{len(df):,} rows | {df.source_file.nunique()} COMSOL exports")
df.head()

## 1. Impurity position, uniform chain

The first ten eigenvalues against the impurity position `x_i` for the three
inter-dot gaps. The ground state stays pinned near the isolated 2D donor:
Coulomb wins over confinement.

In [ ]:
table = modes(case(df, "impurity_position", 0, chain="uniform"))
table.head()

In [ ]:
path = plots.position_uniform(df)
display(Image(filename=str(path)))

## 2. Impurity position, non-uniform chain

Same scan when the fifth dot has `R5 = 10 nm` (left column) and `R5 = 50 nm`
(right column). The asymmetry shifts the oscillation pattern of the modes.

In [ ]:
path = plots.position_nonuniform(df)
display(Image(filename=str(path)))

## 3. Degree of non-uniformity

Energy versus the fifth-dot radius `R5`, with the impurity absent. The avoided
crossing near `R5 = 30 nm` separates the donor regime (`R5 < 30 nm`) from the
acceptor regime (`R5 > 30 nm`).

In [ ]:
path = plots.nonuniformity(df)
display(Image(filename=str(path)))

### With the impurity at the centre

The same scan with the donor at `x_i = 0`. The impurity lowers the spectrum and
changes which modes take part in the avoided crossing.

In [ ]:
path = plots.nonuniformity_impurity(df)
display(Image(filename=str(path)))

## 4. Electric field, uniform chain

The field tilts the chain and the ground state changes character at an
anticrossing. The critical field is inversely proportional to the inter-dot
gap: 1.2, 1.0 and 0.8 kV/cm for `d = -10, 0, 10 nm`.

In [ ]:
path = plots.field_uniform(df)
display(Image(filename=str(path)))

### Induced dipole and critical field

`p = -dE/dF` is fitted on both sides of the anticrossing on the ground state.

In [ ]:
rows = []
for d_nm in (-10, 0, 10):
    table = modes(case(df, "electric_field", d_nm, chain="uniform"))
    fit = fit_two_lines(table.index.to_numpy(float), table["VP_1"].to_numpy(float))
    rows.append({"d_nm": d_nm, **fit})
pd.DataFrame(rows).round(3)

## Takeaways

- The donor pins the ground state near the isolated 2D donor energy, so the
  Coulomb interaction dominates the confinement.
- The non-uniform chain introduces an avoided crossing near `R5 = 30 nm` where
  the fifth dot switches from donor to acceptor.
- The critical field needed to delocalize the electron decreases as the dots
  separate: a more separated chain is easier to polarize.
- The induced-dipole plateaus grow in magnitude with `d`, and their sign flips
  across the anticrossing.